# Machine Learning Model for Server Log Prediction

## Overview
This notebook builds and compares two machine learning models (KNN and Decision Tree) to predict server log outcomes from the `server.csv` dataset.

## Workflow Steps

1. **Data Loading & Exploration**
    - Load the dataset using pandas
    - Display basic information (shape, data types, first rows)
    - Check for missing values and data distribution

2. **Data Preprocessing**
    - Handle missing values (removal or imputation)
    - Encode categorical variables using LabelEncoder or OneHotEncoder
    - Scale numerical features using StandardScaler (important for KNN)
    - Split data into training and testing sets

3. **Model Training**
    - Train a K-Nearest Neighbors (KNN) classifier
    - Train a Decision Tree Classifier
    - Use cross-validation for model validation

4. **Model Evaluation**
    - Compare models using:
      - Accuracy Score
      - Precision, Recall, and F1-Score
      - Confusion Matrix visualization
    - Identify the best-performing model

5. **Prediction Function**
    - Create a function to predict outcomes for new server log entries
    - Use the best model for production predictions

## Libraries Used
- `pandas` - Data manipulation
- `scikit-learn` - Machine learning models and evaluation
- `matplotlib` & `seaborn` - Data visualization
- `numpy` - Numerical operations

## 1. Import Required Libraries

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Machine Learning - Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Machine Learning - Evaluation
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Settings
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

## 2. Data Loading & Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('server.csv')

print("Dataset loaded successfully!")
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"   - Rows: {df.shape[0]:,}")
print(f"   - Columns: {df.shape[1]}")

In [ ]:
# Display first few rows
print("\n📋 First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset information
print("\n🔍 Dataset Information:")
df.info()

In [ ]:
# Statistical summary
print("\n📊 Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("\n🔎 Missing Values Analysis:")
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_percent
})

missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
else:
    print("✅ No missing values found!")

In [ ]:
# Check target variable distribution
print("\n🎯 Target Variable Distribution (is_malicious):")
target_counts = df['is_malicious'].value_counts()
print(target_counts)
print(f"\nClass Distribution:")
print(df['is_malicious'].value_counts(normalize=True) * 100)

# Visualize target distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='is_malicious', palette='viridis')
plt.title('Distribution of Target Variable (is_malicious)', fontsize=14, fontweight='bold')
plt.xlabel('Is Malicious', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0, 1], ['Normal (0)', 'Malicious (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# Check data types
print("\n📝 Column Data Types:")
print(df.dtypes)

## 3. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

print("🔧 Starting data preprocessing...")
print(f"Initial shape: {df_processed.shape}")

In [ ]:
# Handle missing values (if any)
# Strategy: Drop rows with missing values or use imputation

initial_rows = len(df_processed)

# Remove rows with missing values in the target column
df_processed = df_processed.dropna(subset=['is_malicious'])

# For other columns, fill with median (for numerical) or mode (for categorical)
for col in df_processed.columns:
    if df_processed[col].isnull().sum() > 0:
        if df_processed[col].dtype in ['float64', 'int64']:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
        else:
            df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)

rows_removed = initial_rows - len(df_processed)
print(f"✅ Missing values handled: {rows_removed} rows removed")
print(f"New shape: {df_processed.shape}")

In [ ]:
# Identify and separate features by type
# Drop non-predictive columns (timestamp, session_id, source_ip)

# Columns to drop (non-predictive)
cols_to_drop = ['timestamp', 'source_ip', 'session_id']
df_processed = df_processed.drop(columns=[col for col in cols_to_drop if col in df_processed.columns])

print(f"\n🗑️ Dropped non-predictive columns: {cols_to_drop}")
print(f"Remaining columns: {df_processed.shape[1]}")

In [ ]:
# Separate features and target
X = df_processed.drop('is_malicious', axis=1)
y = df_processed['is_malicious']

print(f"\n✅ Features (X) shape: {X.shape}")
print(f"✅ Target (y) shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Handle categorical variables (if any remain)
categorical_cols = X.select_dtypes(include=['object']).columns

if len(categorical_cols) > 0:
    print(f"\n🔤 Encoding categorical columns: {list(categorical_cols)}")
    
    le = LabelEncoder()
    for col in categorical_cols:
        X[col] = le.fit_transform(X[col].astype(str))
    
    print("✅ Categorical encoding complete")
else:
    print("\n✅ No categorical columns to encode")

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("\n📊 Data Split Complete:")
print(f"   Training set: {X_train.shape[0]:,} samples ({(len(X_train)/len(X))*100:.1f}%)")
print(f"   Testing set:  {X_test.shape[0]:,} samples ({(len(X_test)/len(X))*100:.1f}%)")
print(f"\n   Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)
print(f"\n   Testing target distribution:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
# Feature scaling (StandardScaler - important for KNN)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n⚖️ Feature Scaling Complete (StandardScaler)")
print(f"   Mean: ~0, Std: ~1")
print(f"\n   Sample statistics after scaling:")
print(f"   Training mean: {X_train_scaled.mean():.4f}")
print(f"   Training std:  {X_train_scaled.std():.4f}")

## 4. Model Training

### 4.1 K-Nearest Neighbors (KNN) Classifier

In [ ]:
# Initialize and train KNN model
print("\n🤖 Training K-Nearest Neighbors (KNN) Classifier...")

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    metric='euclidean'
)

# Train the model
knn_model.fit(X_train_scaled, y_train)

print("✅ KNN Model trained successfully!")
print(f"   Parameters: n_neighbors=5, weights='uniform', metric='euclidean'")

In [ ]:
# Cross-validation for KNN
print("\n🔄 Performing 5-Fold Cross-Validation for KNN...")

knn_cv_scores = cross_val_score(
    knn_model, 
    X_train_scaled, 
    y_train, 
    cv=5, 
    scoring='accuracy'
)

print(f"\n📊 KNN Cross-Validation Results:")
print(f"   Fold scores: {[f'{score:.4f}' for score in knn_cv_scores]}")
print(f"   Mean CV Accuracy: {knn_cv_scores.mean():.4f} (+/- {knn_cv_scores.std() * 2:.4f})")

### 4.2 Decision Tree Classifier

In [ ]:
# Initialize and train Decision Tree model
print("\n🌳 Training Decision Tree Classifier...")

dt_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

# Train the model (Decision Tree doesn't require scaled data, but we'll use it for consistency)
dt_model.fit(X_train_scaled, y_train)

print("✅ Decision Tree Model trained successfully!")
print(f"   Parameters: max_depth=10, min_samples_split=20, min_samples_leaf=10")

In [ ]:
# Cross-validation for Decision Tree
print("\n🔄 Performing 5-Fold Cross-Validation for Decision Tree...")

dt_cv_scores = cross_val_score(
    dt_model, 
    X_train_scaled, 
    y_train, 
    cv=5, 
    scoring='accuracy'
)

print(f"\n📊 Decision Tree Cross-Validation Results:")
print(f"   Fold scores: {[f'{score:.4f}' for score in dt_cv_scores]}")
print(f"   Mean CV Accuracy: {dt_cv_scores.mean():.4f} (+/- {dt_cv_scores.std() * 2:.4f})")

## 5. Model Evaluation & Comparison

In [ ]:
# Make predictions on test set
knn_predictions = knn_model.predict(X_test_scaled)
dt_predictions = dt_model.predict(X_test_scaled)

print("✅ Predictions generated for both models")

### 5.1 KNN Model Evaluation

In [ ]:
# KNN Evaluation Metrics
print("\n🔍 KNN Model Performance Metrics:")
print("=" * 50)

knn_accuracy = accuracy_score(y_test, knn_predictions)
knn_precision = precision_score(y_test, knn_predictions, zero_division=0)
knn_recall = recall_score(y_test, knn_predictions, zero_division=0)
knn_f1 = f1_score(y_test, knn_predictions, zero_division=0)

print(f"Accuracy:  {knn_accuracy:.4f}")
print(f"Precision: {knn_precision:.4f}")
print(f"Recall:    {knn_recall:.4f}")
print(f"F1-Score:  {knn_f1:.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, knn_predictions, target_names=['Normal', 'Malicious']))

In [ ]:
# KNN Confusion Matrix
knn_cm = confusion_matrix(y_test, knn_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(knn_cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Normal', 'Malicious'],
            yticklabels=['Normal', 'Malicious'])
plt.title('KNN - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

### 5.2 Decision Tree Model Evaluation

In [ ]:
# Decision Tree Evaluation Metrics
print("\n🔍 Decision Tree Model Performance Metrics:")
print("=" * 50)

dt_accuracy = accuracy_score(y_test, dt_predictions)
dt_precision = precision_score(y_test, dt_predictions, zero_division=0)
dt_recall = recall_score(y_test, dt_predictions, zero_division=0)
dt_f1 = f1_score(y_test, dt_predictions, zero_division=0)

print(f"Accuracy:  {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall:    {dt_recall:.4f}")
print(f"F1-Score:  {dt_f1:.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, dt_predictions, target_names=['Normal', 'Malicious']))

In [ ]:
# Decision Tree Confusion Matrix
dt_cm = confusion_matrix(y_test, dt_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(dt_cm, annot=True, fmt='d', cmap='Greens', cbar=True,
            xticklabels=['Normal', 'Malicious'],
            yticklabels=['Normal', 'Malicious'])
plt.title('Decision Tree - Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

### 5.3 Model Comparison

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': ['KNN', 'Decision Tree'],
    'Accuracy': [knn_accuracy, dt_accuracy],
    'Precision': [knn_precision, dt_precision],
    'Recall': [knn_recall, dt_recall],
    'F1-Score': [knn_f1, dt_f1],
    'CV Mean': [knn_cv_scores.mean(), dt_cv_scores.mean()]
})

print("\n📊 Model Comparison Summary:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

In [ ]:
# Visualize comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, comparison_df.iloc[0, 1:5], width, label='KNN', color='#3498db')
rects2 = ax.bar(x + width/2, comparison_df.iloc[1, 1:5], width, label='Decision Tree', color='#2ecc71')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=9)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

In [ ]:
# Identify best model
best_model_idx = comparison_df['F1-Score'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_model_f1 = comparison_df.loc[best_model_idx, 'F1-Score']

print("\n🏆 BEST MODEL SELECTION:")
print("=" * 50)
print(f"Best Model: {best_model_name}")
print(f"F1-Score: {best_model_f1:.4f}")
print("=" * 50)

# Select best model object
if best_model_name == 'KNN':
    best_model = knn_model
    print("\n✅ KNN model selected for production use")
else:
    best_model = dt_model
    print("\n✅ Decision Tree model selected for production use")

## 6. Prediction Function for Production

In [ ]:
def predict_server_log(log_data, model=best_model, scaler_obj=scaler, feature_names=list(X.columns)):
    """
    Predict whether a server log entry is malicious or normal.
    
    Parameters:
    -----------
    log_data : dict or pd.DataFrame
        Server log data with features matching the training data
    model : fitted sklearn model
        The trained model to use for prediction (default: best_model)
    scaler_obj : fitted StandardScaler
        The scaler used during training
    feature_names : list
        List of feature names in the correct order
    
    Returns:
    --------
    dict : Prediction results with probability scores
    """
    # Convert dict to DataFrame if needed
    if isinstance(log_data, dict):
        log_df = pd.DataFrame([log_data])
    else:
        log_df = log_data.copy()
    
    # Ensure all required features are present
    missing_features = set(feature_names) - set(log_df.columns)
    if missing_features:
        raise ValueError(f"Missing required features: {missing_features}")
    
    # Select and order features correctly
    log_df = log_df[feature_names]
    
    # Scale the features
    log_scaled = scaler_obj.transform(log_df)
    
    # Make prediction
    prediction = model.predict(log_scaled)[0]
    prediction_proba = model.predict_proba(log_scaled)[0] if hasattr(model, 'predict_proba') else None
    
    # Prepare result
    result = {
        'prediction': 'Malicious' if prediction == 1 else 'Normal',
        'prediction_value': int(prediction),
        'model_used': best_model_name
    }
    
    if prediction_proba is not None:
        result['confidence'] = {
            'normal': float(prediction_proba[0]),
            'malicious': float(prediction_proba[1])
        }
    
    return result

print("✅ Prediction function created successfully!")
print("\nFunction: predict_server_log(log_data)")
print("Usage: Pass a dictionary or DataFrame with server log features")

### 6.1 Demo: Test Prediction Function

In [ ]:
# Test the prediction function with a sample from the test set
print("\n🧪 Testing Prediction Function:")
print("=" * 60)

# Get a random sample from test set
sample_idx = np.random.randint(0, len(X_test))
sample_data = X_test.iloc[sample_idx].to_dict()
actual_label = y_test.iloc[sample_idx]

# Make prediction
prediction_result = predict_server_log(sample_data)

print(f"\n📝 Sample Log Entry (Index: {sample_idx}):")
for key, value in list(sample_data.items())[:5]:  # Show first 5 features
    print(f"   {key}: {value:.4f}")
print("   ...")

print(f"\n🎯 Prediction Results:")
print(f"   Predicted: {prediction_result['prediction']}")
print(f"   Actual: {'Malicious' if actual_label == 1 else 'Normal'}")
print(f"   Model Used: {prediction_result['model_used']}")

if 'confidence' in prediction_result:
    print(f"\n   Confidence Scores:")
    print(f"      Normal: {prediction_result['confidence']['normal']:.4f}")
    print(f"      Malicious: {prediction_result['confidence']['malicious']:.4f}")

match = "✅ CORRECT" if prediction_result['prediction_value'] == actual_label else "❌ INCORRECT"
print(f"\n   Result: {match}")
print("=" * 60)

## 7. Summary & Conclusions

### Key Findings:

1. **Data Quality**: 
   - The dataset contains server log features for malicious activity detection
   - Features include request patterns, status codes, timing metrics, and network characteristics

2. **Model Performance**:
   - Both KNN and Decision Tree models were trained and evaluated
   - Cross-validation was used to ensure robust performance estimates
   - The best model was selected based on F1-Score

3. **Production Deployment**:
   - A prediction function is ready for production use
   - The function handles preprocessing, scaling, and prediction automatically
   - Provides confidence scores for better decision-making

### Next Steps:

1. **Model Optimization**:
   - Hyperparameter tuning using GridSearchCV or RandomizedSearchCV
   - Feature engineering to improve model performance
   - Try ensemble methods (Random Forest, Gradient Boosting)

2. **Production Deployment**:
   - Save the trained model using joblib or pickle
   - Create an API endpoint for real-time predictions
   - Implement monitoring and logging

3. **Model Maintenance**:
   - Regular retraining with new data
   - Performance monitoring
   - A/B testing for model improvements

In [ ]:
# Optional: Save the best model for future use
import pickle

# Save model
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)
    
print("\n💾 Model artifacts saved successfully!")
print("   - best_model.pkl")
print("   - scaler.pkl")
print("   - feature_names.pkl")
print("\nYou can load these files later for making predictions without retraining.")